# DSP-Prior GR LSTM — differentiable compressor prior + zero-init correction

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub* (`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo and mounts Drive for the dataset. **Push local changes before running.**

Replaces the bins recipe (`train_lstm_tfilm_gr_bins.ipynb`) after its measured val→test gap (0.267 → 0.554 dB GR MAE): the raw-waveform conv frontend can encode song-specific timbre, and the added machinery patched training-dynamics symptoms of that representation problem. This model constrains the frontend to the *detector family* (rectify → smooth → dB — phase- and timbre-blind by construction) and applies the `06_output` gain-prior philosophy one level up: a physical prior with zero-init learned corrections.

```
dry ─ x² ─ frame energy (hop 256) ─ one-pole detector bank (learnable τ) ─ dB
                                     │ ch 0 = prior level L
knobs ─ analytic denorm + zero-init MLP → (T̂, R̂, knee, τ̂a, τ̂r)   ← init = front-panel values
                                     │
g_static = soft-knee gain computer(L; T̂, R̂, knee)      [Giannoulis 2012]
gr_dsp   = attack/release one-pole ballistics(τ̂a, τ̂r)   [frame-rate scan]
                                     │
LSTM( envs ⊕ gr_dsp ⊕ knobs ) → zero-init head → Δgr = 6·tanh(·)
gr = gr_dsp + Δgr                                        ~5.7k params
```

At step 0 the output **is** a textbook feed-forward compressor at the exact knob values (verified by an assert in cell 6) — training can only improve on it. Conditioning is physical: threshold/ratio shift the gain-computer knee directly, attack/release set the ballistics time constants.

**Removed vs the bins notebook** (each was patching the representation problem): 71-bin head + Gaussian soft targets + BCE (→ direct dB regression, no train/eval decode mismatch), LDS reweighting (deep GR now comes from the prior, not a rare-tail fit), CFG conditioning dropout, Fourier knob embedding, TFiLM, and the whole stateful-TBPTT + cold-start dataset stack (→ same 3 s crop regime as the downstream gain-prior model, fresh state per batch, warmup mask). **Kept**: dry-energy-floor masking, Δ-timing loss term, split seed 42.

**Loss**: `Huber(dB) + 1.0·Huber(ΔdB) + 0.1·Huber(gr_dsp, dB)` — the small prior term keeps the DSP path calibrated so its learned parameters stay interpretable (inspection table in cell 10).

No nablafx dependency — cell 0 is plain torch + lightning.

In [ ]:
# ── 0. Dependencies ──────────────────────────────────────────────────
# No nablafx needed (and therefore no rational/frechet stubs). Pin numpy
# first so lightning installs can't downgrade Colab's numpy 2.x and break
# torch; install lightning --no-deps so it can't clobber Colab's CUDA torch.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile lightning-utilities packaging
!pip install -q --no-deps lightning

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} — restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

In [ ]:
# ── 1. Mount Drive (dataset) + clone repo from GitHub (code) ─────────
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub —
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" pull --ff-only
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT
COND_DIR = os.path.join(REPO_ROOT, "05_conditioning")
OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "gr_pred_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isfile(os.path.join(COND_DIR, "model_dsp_prior.py")), (
    f"Clone failed or stale: {REPO_ROOT}. Did you push local changes?"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for p in (REPO_ROOT, COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# ── 2. Cache dataset to Colab local SSD ──────────────────────────────

import shutil
from dataset import discover_diffssl_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_diffssl_gr_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs × {len(settings)} settings → {LOCAL_DATA_ROOT}")

local_dry = Path(LOCAL_DATA_ROOT) / "processed_normalized"
local_dry.mkdir(parents=True, exist_ok=True)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    src = Path(DATA_ROOT) / "processed_normalized" / fn
    dst = local_dry / fn
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)

for setting in settings:
    local_gr = Path(LOCAL_DATA_ROOT) / "gr_curves" / setting
    local_gr.mkdir(parents=True, exist_ok=True)
    for song in songs:
        fn = f"{song}.pt"
        src = Path(DATA_ROOT) / "gr_curves" / setting / fn
        if not src.is_file():
            continue
        dst = local_gr / fn
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# ── 3. Imports & hyper-parameters ────────────────────────────────────

import json
from datetime import datetime

import torch
import torch.nn.functional as F
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset_crops import (
    BATCH_SIZE, SAMPLE_LENGTH, SAMPLE_RATE, GRPredCropDataModule,
)
from model_dsp_prior import DSPPriorGRLSTM
from system_dsp_prior import DSPPriorGRSystem
from splits import DIFFSSL_PARAM_RANGES, build_split_manifest, normalize_setting_params
from gr_target import GR_DB_MAX, GR_DB_MIN
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split (identical to the bins notebook / 02b / 06_output) --
SPLIT_SEED = 42
N_VAL_SONGS = 1
N_TEST_SONGS = 2

# -- training: 3 s crops, fresh state per batch, fixed cosine budget --
LR = 1e-3
MAX_EPOCHS = 150
SCHEDULER = "cosine"
ETA_MIN = 1e-6

HOP_SIZE = 256
WARMUP_SEC = 1.0          # ≥ longest release (0.8 s): crop-start GR is ambiguous
WARMUP_FRAMES = int(WARMUP_SEC * SAMPLE_RATE / HOP_SIZE)

# -- loss --
ENERGY_FLOOR_DB = -60.0   # frames with dry RMS below this are masked (noise labels)
HUBER_BETA_DB = 1.0       # quadratic below 1 dB error, linear above
DELTA_WEIGHT = 1.0        # first-difference term (attack/release timing)
PRIOR_WEIGHT = 0.1        # keeps gr_dsp anchored to the target (interpretability)

# -- model --
HIDDEN_SIZE = 32
DETECTOR_TAUS_MS = (12.0, 2.0, 40.0, 150.0)  # ch 0 (≈ RMS-1024 avg delay) feeds the prior
DETECTOR_KERNEL_FRAMES = 256                  # 1.5 s causal context
DELTA_MAX_DB = 6.0                            # bound on the learned GR correction
KNEE_DB_INIT = 6.0
KNOB_HIDDEN = 16

print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | hop {HOP_SIZE} "
      f"({SAMPLE_RATE/HOP_SIZE:.0f} Hz frames) | warmup {WARMUP_FRAMES} frames")

RUN_TAG = "lstm_dsp_prior_gr"
RESUME_RUN = None

In [ ]:
# ── 4. Preview split (must match the bins notebook / 02b / 06) ───────

preview = build_split_manifest(
    discover_diffssl_gr_pairs(DATA_ROOT),
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(
    f"Pairs — train={len(preview.train_pair_keys)} "
    f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}"
)

In [ ]:
# ── 5. Model size ────────────────────────────────────────────────────

model = DSPPriorGRLSTM(
    hop_size=HOP_SIZE,
    sample_rate=SAMPLE_RATE,
    detector_taus_ms=DETECTOR_TAUS_MS,
    detector_kernel_frames=DETECTOR_KERNEL_FRAMES,
    hidden_size=HIDDEN_SIZE,
    delta_max_db=DELTA_MAX_DB,
    knee_db_init=KNEE_DB_INIT,
    knob_hidden=KNOB_HIDDEN,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"DSPPriorGRLSTM: {n_params:,} params (bins model was 11.7k)")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")

# knob map init sanity: DSP params == front-panel values before training
with torch.no_grad():
    _p = torch.tensor([normalize_setting_params("threshold_-12_attack_10_release_0.4_ratio_10")])
    _d = model.knob_map(_p)
print("init check (T=-12, a=10ms, r=0.4s, R=10):",
      f"T̂={_d['threshold_db'][0]:.2f} dB, τ̂a={_d['att_s'][0]*1e3:.1f} ms, "
      f"τ̂r={_d['rel_s'][0]:.2f} s, R̂={_d['ratio'][0]:.1f}, knee={_d['knee_db'][0]:.1f} dB")

In [ ]:
# ── 6. DataModule + zero-init sanity check ───────────────────────────
# The zero-init head makes the untrained model IDENTICAL to the pure DSP
# compressor at the front-panel knob values. Verify on a real batch and
# report that baseline's GR MAE — training starts there, not from noise.

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"lstm_gr_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = GRPredCropDataModule(
    data_root=DATA_ROOT,
    sample_length=SAMPLE_LENGTH,
    sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path,
    num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

# -- zero-init sanity: untrained model == pure DSP compressor GR --------
if not RESUME_RUN:
    _dry, _gr, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        _pred, _prior, _delta = model(_dry, _p, return_parts=True)
    assert float(_delta.abs().max()) == 0.0, "correction head is not zero-initialised!"
    _n = _pred.shape[-1] * HOP_SIZE
    _tgt = F.avg_pool1d(_gr[..., :_n], HOP_SIZE)
    _en = 10 * torch.log10(F.avg_pool1d(_dry[..., :_n] ** 2, HOP_SIZE) + 1e-12) > ENERGY_FLOOR_DB
    _en[..., :WARMUP_FRAMES] = False
    print(f"zero-init |Δgr| max = {float(_delta.abs().max()):.2e}")
    print(f"untrained (== DSP prior) val-crop GR MAE: {float((_pred - _tgt).abs()[_en].mean()):.3f} dB"
          "  <- training starts here")
    del _dry, _gr, _p, _pred, _prior, _delta

In [ ]:
# ── 7. Train ─────────────────────────────────────────────────────────

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gr_prediction_dsp_prior",
        "model_type": "dsp_prior_gr_lstm",
        "dataset": "Diff-SSL-G-Comp",
        "setting": "multi (10 settings)",
        "conditioning": "knob_to_dsp_params (physical, zero-init corr) + knob_concat",
        "frontend": "one_pole_detector_bank (level-domain, no raw-waveform conv)",
        "sample_rate": SAMPLE_RATE,
        "hop_size": HOP_SIZE,
        "sample_length": SAMPLE_LENGTH,
        "batch_size": BATCH_SIZE,
        "training": "3s_crops_fresh_state_per_batch (no TBPTT, no cold-start mixing)",
        "gr_range_db": [GR_DB_MIN, GR_DB_MAX],
        "param_order": PARAM_ORDER,
        "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "test_settings": dm.split.test_settings,
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs,
        "test_songs": dm.split.test_songs,
        "loss": {
            "kind": "masked_huber_delta_prior",
            "energy_floor_db": ENERGY_FLOOR_DB,
            "huber_beta_db": HUBER_BETA_DB,
            "delta_weight": DELTA_WEIGHT,
            "prior_weight": PRIOR_WEIGHT,
            "warmup_frames": WARMUP_FRAMES,
        },
        "model": {
            "hidden_size": HIDDEN_SIZE,
            "detector_taus_ms_init": list(DETECTOR_TAUS_MS),
            "detector_kernel_frames": DETECTOR_KERNEL_FRAMES,
            "delta_max_db": DELTA_MAX_DB,
            "knee_db_init": KNEE_DB_INIT,
            "knob_hidden": KNOB_HIDDEN,
            "num_params": n_params,
        },
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "scheduler": SCHEDULER,
        "eta_min": ETA_MIN,
    }, f, indent=2)

system = DSPPriorGRSystem(
    model=model,
    lr=LR,
    warmup_frames=WARMUP_FRAMES,
    energy_floor_db=ENERGY_FLOOR_DB,
    huber_beta_db=HUBER_BETA_DB,
    delta_weight=DELTA_WEIGHT,
    prior_weight=PRIOR_WEIGHT,
    scheduler=SCHEDULER,
    max_epochs=MAX_EPOCHS,
    eta_min=ETA_MIN,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(
        dirpath=ckpt_dir,
        monitor="loss/val",
        mode="min",
        save_top_k=3,
        save_last=True,
        filename="best-{epoch:03d}-{step}",
        auto_insert_metric_name=False,
    ),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=callbacks,
    logger=loggers,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
)

trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

In [ ]:
# ── 8. Test (held-out songs × lowest-threshold settings) ─────────────

trainer.test(system, datamodule=dm, ckpt_path=callbacks[0].best_model_path)

In [ ]:
# ── 9. Full-song streaming eval + plot ───────────────────────────────
# Streams whole val/test songs chunked with explicit state carry (the
# deployment path — chunked forward is exact vs one full forward). The
# printed full-context GR MAE is directly comparable to the run table.

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
model_e = system.model
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

CHUNK = int(10.0 * SAMPLE_RATE) // HOP_SIZE * HOP_SIZE


@torch.no_grad()
def stream_pair(m):
    audio, _ = sf.read(m["dry"], dtype="float32", always_2d=True)
    dry = torch.from_numpy(audio.T)
    if dry.shape[0] > 1:
        dry = dry.mean(dim=0, keepdim=True)
    gr = torch.load(m["gr"], weights_only=False)["gr_db"].float()
    if gr.dim() == 1:
        gr = gr.unsqueeze(0)
    n = min(dry.shape[-1], gr.shape[-1])
    n -= n % HOP_SIZE
    dry, gr = dry[None, ..., :n], gr[None, ..., :n]
    params = torch.tensor(m["params"], dtype=torch.float32)[None].cuda()

    state, preds, priors = None, [], []
    for o in range(0, n, CHUNK):
        p, pr, _, state = model_e(
            dry[..., o : o + CHUNK].cuda(), params, state,
            return_state=True, return_parts=True,
        )
        preds.append(p.cpu())
        priors.append(pr.cpu())
    pred, prior = torch.cat(preds, -1), torch.cat(priors, -1)
    tgt = F.avg_pool1d(gr, HOP_SIZE)
    en = 10 * torch.log10(F.avg_pool1d(dry**2, HOP_SIZE) + 1e-12) > ENERGY_FLOOR_DB
    return pred, prior, tgt, en


for split_name in ("val", "test"):
    errs, wsum = 0.0, 0
    for m in dm.meta[split_name]:
        pred, prior, tgt, en = stream_pair(m)
        e = (pred - tgt).abs()[en]
        errs += float(e.sum())
        wsum += int(en.sum())
        print(f"  {split_name} {m['song']:16s} {m['setting']:45s} "
              f"MAE {float(e.mean()):.3f} dB (prior {float((prior - tgt).abs()[en].mean()):.3f})")
    print(f"{split_name.upper()} full-context GR MAE: {errs / max(wsum, 1):.3f} dB\n")

# -- plot: mid-song window of a val pair, prediction vs prior vs target --
pick = dm.meta["val"][len(dm.meta["val"]) // 2]
pred, prior, tgt, en = stream_pair(pick)
Tf = pred.shape[-1]
w0, w1 = Tf // 2, Tf // 2 + int(15.0 * SAMPLE_RATE / HOP_SIZE)  # 15 s window
t = np.arange(w0, w1) * HOP_SIZE / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, tgt[0, 0, w0:w1], label="Target GR", alpha=0.8, lw=0.7)
ax.plot(t, pred[0, 0, w0:w1], label="Predicted GR", alpha=0.8, lw=0.7)
ax.plot(t, prior[0, 0, w0:w1], label="DSP prior (gr_dsp)", alpha=0.6, lw=0.7, ls="--")
l1 = float((pred - tgt).abs()[..., w0:w1][en[..., w0:w1]].mean())
ax.set_title(f"{pick['song']} / {pick['setting']} — window L1 = {l1:.2f} dB")
ax.set_xlabel("Time (s)")
ax.set_ylabel("GR (dB)")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_gr_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
# ── 10. Learned DSP parameters vs front-panel values ─────────────────
# The prior term keeps these interpretable: how far did training pull the
# effective threshold/ratio/ballistics away from the knob labels? Deviations
# are the model's estimate of detector calibration + feedback-topology skew.

print(f"detector τ (ms): init {list(DETECTOR_TAUS_MS)} -> "
      f"learned {[round(float(v), 1) for v in model_e.detector.taus_ms]}")
print(f"{'setting':46s} {'T̂ dB':>8s} {'R̂':>6s} {'knee':>6s} {'τ̂a ms':>7s} {'τ̂r s':>6s}")
with torch.no_grad():
    for s in dm.split.all_settings:
        p = torch.tensor([normalize_setting_params(s)], device="cuda")
        d = model_e.knob_map(p)
        print(f"{s:46s} {float(d['threshold_db']):8.2f} {float(d['ratio']):6.2f} "
              f"{float(d['knee_db']):6.2f} {float(d['att_s']) * 1e3:7.2f} "
              f"{float(d['rel_s']):6.3f}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"